# Day 09: Dropout: A Simple Way to Prevent Neural Networks from Overfitting (PyTorch Implementation)

> **Bài báo gốc:** *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*  
> **Tác giả:** Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever, Ruslan Salakhutdinov  
> **Xuất bản:** Journal of Machine Learning Research (JMLR), 15(56):1929-1958, 2014  
> **Tài liệu gốc:** [Day 09 Dropout.pdf](./resources/Day%2009%20Dropout.pdf)

---

## 📌 Giới Thiệu Chuyên Sâu Cho Người Mới Bắt Đầu

Trong học sâu cổ điển (kỷ nguyên trước 2012), các mô hình mạng nơ-ron sâu thường xuyên rơi vào thảm họa **Quá Khớp (Overfitting)** khi số lượng tham số lớn hơn rất nhiều số lượng dữ liệu huấn luyện ($P \gg N$). 
- Khi đó, các nơ-ron xuất hiện hiện tượng **Đồng Thích Ứng Phức Tạp (Complex Co-adaptation)**: Nơ-ron $A$ mắc lỗi, nơ-ron $B$ thay vì học đặc trưng tổng quát lại học cách bù trừ cho lỗi của $A$ trên đúng tập dữ liệu huấn luyện đó.
- Kỹ thuật điều hòa truyền thống thời bấy giờ là **$L_2$ Weight Decay** chỉ phạt độ lớn của trọng số (thu nhỏ bán kính siêu cầu) nhưng **hoàn toàn bất lực** trong việc phá vỡ sự phụ thuộc chéo giữa các nơ-ron.

Đột phá của Srivastava, Hinton cùng các cộng sự năm 2014 với **Dropout** đã giải quyết triệt để vấn đề này:
1. **Triệt tiêu ngẫu nhiên nơ-ron:** Với xác suất giữ lại $p_{keep} = 1 - p$, mỗi nơ-ron bị tắt ngẫu nhiên trong từng lượt huấn luyện (Mini-batch).
2. **Phá vỡ đồng thích ứng:** Vì một nơ-ron không bao giờ biết trước nơ-ron bên cạnh có bị tắt hay không, nó buộc phải tự lực học các đặc trưng mạnh mẽ và hữu ích độc lập.
3. **Mô hình tổ hợp mũ (Exponential Ensemble):** Dropout tương đương với việc huấn luyện một tổ hợp gồm $2^N$ mạng nơ-ron con khác nhau cùng chia sẻ trọng số, và tính trung bình hình học các dự đoán ở thời điểm kiểm thử.


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[✓] Đã nạp thành công PyTorch phiên bản: {torch.__version__} | Thiết bị: {device}")
except ImportError:
    HAS_TORCH = False
    print("[!] Không tìm thấy PyTorch trong môi trường. Vui lòng cài đặt: pip install torch")


## 1. Cơ Chế Inverted Dropout Trong PyTorch

Trong bài báo gốc (2014), nhóm tác giả áp dụng **Standard Dropout**:
- **Train:** $y = x \odot \text{mask}$, với $\text{mask} \sim \text{Bernoulli}(1 - p)$
- **Test:** $y = x \cdot (1 - p)$ (Phải nhân thu nhỏ trọng số hoặc kích hoạt ở mọi tầng khi triển khai inference)

Tuy nhiên, trong các thư viện hiện đại như **PyTorch** và **TensorFlow**, chuẩn mực thiết kế là **Inverted Dropout**:
- **Train:** $y = \frac{x \odot \text{mask}}{1 - p}$ (Nhân phóng đại trước cho $\frac{1}{1 - p}$ ngay trong lúc train)
- **Test:** $y = x$ (Hoàn toàn giữ nguyên tín hiệu, suy luận **Zero Overhead** $O(1)$)

Dưới đây ta tự cài đặt một toán tử `CustomDropoutFunction` bằng `torch.autograd.Function` để quan sát chính xác forward và backward pass.


In [ ]:
if HAS_TORCH:
    class CustomDropoutFunction(torch.autograd.Function):
        """
        Triển khai Inverted Dropout từ gốc bằng PyTorch Autograd:
        - Forward: Nhân tensor mặt nạ Bernoulli và chia cho (1 - p)
        - Backward: Đạo hàm chỉ truyền ngược qua các nơ-ron còn sống (mask = 1)
        """
        @staticmethod
        def forward(ctx, x, p=0.5, training=True):
            if not training or p <= 0.0:
                ctx.mask = None
                return x
            p_keep = 1.0 - p
            mask = (torch.rand_like(x) < p_keep).float()
            out = (x * mask) / p_keep
            ctx.save_for_backward(mask)
            ctx.p_keep = p_keep
            return out

        @staticmethod
        def backward(ctx, grad_output):
            if ctx.mask is None and not ctx.saved_tensors:
                return grad_output, None, None
            (mask,) = ctx.saved_tensors
            p_keep = ctx.p_keep
            grad_input = (grad_output * mask) / p_keep
            return grad_input, None, None

    # Thử nghiệm trực quan
    dummy_input = torch.ones(2, 6)
    print("Tensor đầu vào gốc:")
    print(dummy_input.numpy())

    out_train = CustomDropoutFunction.apply(dummy_input, 0.5, True)
    print("\nĐầu ra trong pha Huấn luyện (Train mode, p=0.5, scaled x2):")
    print(out_train.numpy())

    out_eval = CustomDropoutFunction.apply(dummy_input, 0.5, False)
    print("\nĐầu ra trong pha Kiểm thử (Eval mode, giữ nguyên 1.0):")
    print(out_eval.numpy())


## 2. Thiết Lập Bài Toán Thực Nghiệm Gây Quá Khớp Cực Cao

Để chứng minh sức mạnh của Dropout so với các phương pháp tiền bối, ta xây dựng một bài toán có nguy cơ quá khớp rất lớn:
- **Kích thước mẫu huấn luyện nhỏ:** $N_{train} = 120$ mẫu.
- **Số chiều đặc trưng lớn:** $D = 40$ chiều.
- **Tín hiệu ngụy tạo (Spurious Correlation):** Chỉ có **2 chiều đầu tiên** chứa tín hiệu phân loại thực tế ($1.5 x_0 - 2.0 x_1 > 0$), còn **38 chiều còn lại là nhiễu trắng thuần túy**.
- **Mục tiêu đối chuẩn:** Mạng sâu không có điều hòa sẽ lập tức học vẹt 38 chiều nhiễu để đạt $100\%$ độ chính xác trên tập Train nhưng thất bại thảm hại trên tập Test độc lập ($N_{test} = 200$).


In [ ]:
def generate_dataset(n_train=120, n_test=200, n_features=40, seed=42):
    np.random.seed(seed)
    
    # Tập huấn luyện
    X_train = np.random.randn(n_train, n_features).astype(np.float32)
    sig_train = 1.5 * X_train[:, 0] - 2.0 * X_train[:, 1] + 0.4 * np.random.randn(n_train)
    y_train = (sig_train > 0).astype(np.int64)

    # Tập kiểm tra độc lập
    X_test = np.random.randn(n_test, n_features).astype(np.float32)
    sig_test = 1.5 * X_test[:, 0] - 2.0 * X_test[:, 1] + 0.4 * np.random.randn(n_test)
    y_test = (sig_test > 0).astype(np.int64)

    return X_train, y_train, X_test, y_test

X_train_np, y_train_np, X_test_np, y_test_np = generate_dataset()
print(f"[+] Kích thước dữ liệu: Train = {X_train_np.shape}, Test = {X_test_np.shape}")
print(f"    Tỉ lệ nhãn dương (Class 1) trên tập Train: {np.mean(y_train_np):.2%}")


## 3. Định Nghĩa 3 Mô Hình Đối Chiếu Giá Trị Học Thuật

Ta thiết kế các mô hình PyTorch tương đương về dung lượng tham số để đối chứng:
1. **VanillaMLP (Tiền-Dropout):** Mạng MLP 3 tầng không có điều hòa.
2. **L2RegularizedMLP (Kỷ nguyên cổ điển):** Cùng kiến trúc MLP, huấn luyện với `weight_decay = 0.02`.
3. **DropoutMLP (Srivastava 2014):** Cùng kiến trúc MLP, chèn lớp `nn.Dropout(p = 0.5)` sau mỗi tầng ReLU ẩn.


In [ ]:
if HAS_TORCH:
    class VanillaMLP(nn.Module):
        def __init__(self, in_features=40, hidden_dims=[64, 32], num_classes=2):
            super().__init__()
            self.fc1 = nn.Linear(in_features, hidden_dims[0])
            self.relu1 = nn.ReLU()
            self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
            self.relu2 = nn.ReLU()
            self.fc_out = nn.Linear(hidden_dims[1], num_classes)

        def forward(self, x):
            h1 = self.relu1(self.fc1(x))
            h2 = self.relu2(self.fc2(h1))
            return self.fc_out(h2)

        def get_hidden_activations(self, x):
            with torch.no_grad():
                return self.relu1(self.fc1(x))

    class DropoutMLP(nn.Module):
        def __init__(self, in_features=40, hidden_dims=[64, 32], num_classes=2, dropout_p=0.5):
            super().__init__()
            self.fc1 = nn.Linear(in_features, hidden_dims[0])
            self.relu1 = nn.ReLU()
            self.drop1 = nn.Dropout(p=dropout_p)
            self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
            self.relu2 = nn.ReLU()
            self.drop2 = nn.Dropout(p=dropout_p)
            self.fc_out = nn.Linear(hidden_dims[1], num_classes)

        def forward(self, x):
            h1 = self.drop1(self.relu1(self.fc1(x)))
            h2 = self.drop2(self.relu2(self.fc2(h1)))
            return self.fc_out(h2)

        def get_hidden_activations(self, x):
            was_training = self.training
            self.eval()
            with torch.no_grad():
                h1 = self.relu1(self.fc1(x))
            if was_training:
                self.train()
            return h1

    print("[✓] Đã khởi tạo các lớp kiến trúc mạng PyTorch thành công!")


## 4. Huấn Luyện & Đối Chuẩn Giá Trị Học Thuật

Tiến hành huấn luyện cả 3 mô hình trong 120 epochs. Đồng thời đo lường:
1. **Độ chính xác Train/Test & Overfitting Gap**: $\Delta = \text{Acc}_{train} - \text{Acc}_{test}$
2. **Chỉ số Đồng thích ứng nơ-ron (Neuron Co-adaptation Metric)**:
$$\text{Mean } |\text{Corr}| = \frac{1}{D(D-1)} \sum_{i \neq j} |\text{Corr}(h_i, h_j)|$$


In [ ]:
if HAS_TORCH:
    # Chuyển đổi tensor sang thiết bị tính toán
    X_train_t = torch.tensor(X_train_np, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train_np, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test_np, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test_np, dtype=torch.long).to(device)

    # Khởi tạo mô hình & optimizer
    torch.manual_seed(42)
    m_vanilla = VanillaMLP().to(device)
    m_l2 = VanillaMLP().to(device)
    m_dropout = DropoutMLP(dropout_p=0.5).to(device)

    criterion = nn.CrossEntropyLoss()
    lr = 0.02
    opt_vanilla = optim.Adam(m_vanilla.parameters(), lr=lr, weight_decay=0.0)
    opt_l2 = optim.Adam(m_l2.parameters(), lr=lr, weight_decay=0.02)
    opt_dropout = optim.Adam(m_dropout.parameters(), lr=lr, weight_decay=0.0)

    models = {
        "Vanilla (No Reg)": (m_vanilla, opt_vanilla),
        "L2 Weight Decay": (m_l2, opt_l2),
        "Dropout (p = 0.5)": (m_dropout, opt_dropout),
    }

    history = {k: {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []} for k in models}

    for epoch in range(120):
        for name, (model, optimizer) in models.items():
            model.train()
            optimizer.zero_grad()
            out = model(X_train_t)
            loss = criterion(out, y_train_t)
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                tr_acc = (torch.argmax(model(X_train_t), dim=1) == y_train_t).float().mean().item() * 100.0
                te_out = model(X_test_t)
                te_loss = criterion(te_out, y_test_t).item()
                te_acc = (torch.argmax(te_out, dim=1) == y_test_t).float().mean().item() * 100.0

            history[name]["train_loss"].append(loss.item())
            history[name]["test_loss"].append(te_loss)
            history[name]["train_acc"].append(tr_acc)
            history[name]["test_acc"].append(te_acc)

    # Đo lường hệ số tương quan đồng thích ứng
    def compute_coadaptation(activations):
        N, D = activations.shape
        centered = activations - np.mean(activations, axis=0, keepdims=True)
        std = np.std(activations, axis=0, keepdims=True) + 1e-8
        normed = centered / std
        corr = np.dot(normed.T, normed) / N
        mask = ~np.eye(D, dtype=bool)
        return np.mean(np.abs(corr[mask])), corr

    corrs, corr_mats = {}, {}
    for name, (model, _) in models.items():
        h = model.get_hidden_activations(X_test_t).cpu().numpy()
        mean_c, c_mat = compute_coadaptation(h)
        corrs[name] = mean_c
        corr_mats[name] = c_mat

    print("=" * 85)
    header = f"{'Mô Hình Nghiên Cứu':<22} | {'Train Acc':<10} | {'Test Acc':<10} | {'Overfit Gap':<12} | {'Co-adaptation |Corr|':<20}"
    print(header)
    print("-" * 85)
    for name in models:
        tr = history[name]["train_acc"][-1]
        te = history[name]["test_acc"][-1]
        gap = tr - te
        c = corrs[name]
        note = "Học vẹt 100%" if gap > 20 else ("Vẫn co-adapt" if gap > 10 else "Tối ưu, độc lập")
        print(f"{name:<22} | {tr:6.1f}%    | {te:6.1f}%    | {gap:6.1f}%      | {c:6.3f} ({note})")
    print("=" * 85)


## 5. Ứng Dụng Hiện Đại: Monte Carlo Dropout (Gal & Ghahramani, 2016)

Trong bài báo tại **ICML 2016**, Yarin Gal và Zoubin Ghahramani đã chứng minh rằng: **Dropout được áp dụng trong thời gian suy luận (Test-time) toán học tương đương với xấp xỉ biến phân (Variational Inference) trong mô hình Gaussian Process!**

Bằng cách cho mẫu kiểm thử chạy qua mạng nhiều lần với `model.train()` (hoặc giữ nguyên Dropout mask ngẫu nhiên), ta thu được một phân phối dự đoán:
- **Kỳ vọng:** $\mathbb{E}[p(y=1|x)] = \frac{1}{T} \sum_{t=1}^T p_t$
- **Độ bất định nhận thức (Epistemic Uncertainty):** $\sigma = \text{Std}(p_t)$ phản ánh mức độ thiếu tự tin của mô hình trước các mẫu nhiễu hoặc nằm gần biên quyết định.


In [ ]:
if HAS_TORCH:
    # Thực hiện Monte Carlo Dropout với 50 lần forward stochastic
    m_dropout.train() # Kích hoạt Dropout trong lúc suy luận
    mc_preds = []
    with torch.no_grad():
        for _ in range(50):
            probs = torch.softmax(m_dropout(X_test_t), dim=1)[:, 1].cpu().numpy()
            mc_preds.append(probs)

    mc_preds = np.array(mc_preds) # Shape: (50, 200)
    mc_mean = np.mean(mc_preds, axis=0)
    mc_std = np.std(mc_preds, axis=0) # Độ bất định

    print(f"[+] Monte Carlo Dropout (50 passes):")
    print(f"    - Độ bất định trung bình trên toàn tập kiểm tra: {np.mean(mc_std):.4f}")
    print(f"    - Mẫu có độ tự tin cao nhất (Uncertainty thấp nhất): {np.min(mc_std):.4f}")
    print(f"    - Mẫu có độ bất định cao nhất (Gần biên quyết định): {np.max(mc_std):.4f}")


## 6. Trực Quan Hóa Đối Chuẩn Học Thuật Toàn Diện

Biểu đồ 4 bảng con phản ánh toàn cảnh giá trị học thuật:
1. **Đường cong mất mát (Loss Explosion):** Vanilla MLP phân kỳ mất mát trên tập Test (Overfitting nặng), trong khi Dropout duy trì độ ổn định tuyệt đối.
2. **Khoảng cách Quá Khớp (Overfitting Gap):** Dropout thu hẹp khoảng cách xuống mức tối thiểu.
3. **Ma trận tương quan nơ-ron ẩn:** Nửa trên (Vanilla) nơ-ron liên kết phụ thuộc cao, nửa dưới (Dropout) các nơ-ron học độc lập.
4. **Phân phối độ bất định Monte Carlo Dropout:** Phân loại rõ ràng độ tin cậy của các điểm dữ liệu kiểm tra.


In [ ]:
if HAS_TORCH and HAS_MATPLOTLIB:
    os.makedirs("outputs", exist_ok=True)
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    fig.suptitle("DROPOUT: A SIMPLE WAY TO PREVENT NEURAL NETWORKS FROM OVERFITTING\nAcademic Value Benchmark & Diagnostic Suite",
                 fontsize=14, fontweight="bold", y=0.98)

    # Panel 1: Loss curves
    ax1 = axes[0, 0]
    ep_range = range(1, 121)
    ax1.plot(ep_range, history["Vanilla (No Reg)"]["train_loss"], 'r--', alpha=0.6, label="Vanilla Train Loss")
    ax1.plot(ep_range, history["Vanilla (No Reg)"]["test_loss"], 'r-', linewidth=2, label="Vanilla Test Loss (Overfitting!)")
    ax1.plot(ep_range, history["Dropout (p = 0.5)"]["train_loss"], 'b--', alpha=0.6, label="Dropout Train Loss")
    ax1.plot(ep_range, history["Dropout (p = 0.5)"]["test_loss"], 'b-', linewidth=2, label="Dropout Test Loss (Stable)")
    ax1.set_title("1. Training vs Testing Loss (Overfitting Explosion)", fontweight="bold")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Cross-Entropy Loss")
    ax1.set_ylim(0, 2.0)
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend(fontsize=9)

    # Panel 2: Accuracy & Gap
    ax2 = axes[0, 1]
    model_names = list(models.keys())
    tr_accs = [history[m]["train_acc"][-1] for m in model_names]
    te_accs = [history[m]["test_acc"][-1] for m in model_names]
    gaps = [tr_accs[i] - te_accs[i] for i in range(len(model_names))]
    x_idx = np.arange(len(model_names))
    width = 0.25
    ax2.bar(x_idx - width, tr_accs, width, label="Train Accuracy (%)", color="#e74c3c", alpha=0.85)
    ax2.bar(x_idx, te_accs, width, label="Test Accuracy (%)", color="#2ecc71", alpha=0.85)
    ax2.bar(x_idx + width, gaps, width, label="Overfitting Gap (%)", color="#f39c12", alpha=0.85)
    ax2.set_title("2. Generalization & Overfitting Gap Comparison", fontweight="bold")
    ax2.set_xticks(x_idx)
    ax2.set_xticklabels(model_names, fontweight="bold")
    ax2.set_ylabel("Accuracy / Gap (%)")
    ax2.set_ylim(0, 115)
    ax2.grid(axis='y', linestyle="--", alpha=0.5)
    ax2.legend(fontsize=9, loc="upper right")

    # Panel 3: Correlation Matrix
    ax3 = axes[1, 0]
    mat_van = corr_mats["Vanilla (No Reg)"][:20, :20]
    mat_dr = corr_mats["Dropout (p = 0.5)"][:20, :20]
    comb_mat = np.triu(mat_van) + np.tril(mat_dr, -1)
    im = ax3.imshow(np.abs(comb_mat), cmap="magma", vmin=0, vmax=0.6)
    ax3.set_title("3. Co-adaptation: Vanilla (Upper) vs Dropout (Lower)", fontweight="bold")
    ax3.set_xlabel("Neuron Index (0-19)")
    ax3.set_ylabel("Neuron Index (0-19)")
    plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

    # Panel 4: MC Dropout Uncertainty Histogram
    ax4 = axes[1, 1]
    ax4.hist(mc_std, bins=25, color="#9b59b6", edgecolor="black", alpha=0.75)
    ax4.axvline(np.mean(mc_std), color="red", linestyle="--", linewidth=2,
                label=f"Mean Uncertainty: {np.mean(mc_std):.3f}")
    ax4.set_title("4. MC Dropout Uncertainty Distribution (Gal 2016)", fontweight="bold")
    ax4.set_xlabel("Predictive Uncertainty (Standard Deviation)")
    ax4.set_ylabel("Number of Test Samples")
    ax4.grid(True, linestyle="--", alpha=0.5)
    ax4.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig("outputs/dropout_academic_comparison.png", dpi=300)
    plt.show()
    print("[✓] Đã xuất biểu đồ đối chuẩn học thuật tại: outputs/dropout_academic_comparison.png")


## 7. Tổng Kết & Bài Học Cho Người Mới Bắt Đầu

1. **Hiểu bản chất của Quá Khớp:** Khi mạng có quá nhiều bậc tự do, các nơ-ron phối hợp ngầm để khớp các đặc tính ngẫu nhiên của tập huấn luyện (Co-adaptation).
2. **Tại sao L2 Weight Decay không đủ:** L2 chỉ kéo co các trọng số về 0, nhưng nếu một nơ-ron mang giá trị dương lớn và nơ-ron khác mang giá trị âm lớn để bù trừ, tích vô hướng vẫn cân bằng và hiện tượng đồng thích ứng vẫn tồn tại.
3. **Bản chất của Dropout:** Dropout cưỡng bức tính độc lập. Bằng cách vô hiệu hóa ngẫu nhiên một nửa số nơ-ron, mỗi nơ-ron phải tạo ra những biểu diễn đặc trưng độc lập và mạnh mẽ.
4. **Quy tắc vàng khi sử dụng PyTorch:**
   - Luôn luôn gọi `model.train()` trước khi huấn luyện (để kích hoạt Dropout).
   - Luôn luôn gọi `model.eval()` trước khi đánh giá hoặc đưa mô hình vào triển khai sản xuất (để tắt Dropout và sử dụng toàn bộ dung lượng mạng).
